# Assignment 3: Time Series Prediction of Sepsis

Authors:  
Max Masuch  
Ismail Mohammed

## Imports

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas import read_csv
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, Conv1D, BatchNormalization, MaxPooling1D, Input
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, multilabel_confusion_matrix, accuracy_score, roc_auc_score

## Load data

This section should load the raw dataset for the task.  
Remember to use relative paths to load any files in the notebook.

In [20]:
df_a = pd.read_csv('raw_data/sepsisexp_timeseries_partition-A.tsv', sep='\t')

df_b = pd.read_csv('raw_data/sepsisexp_timeseries_partition-B.tsv', sep='\t')

df_c = pd.read_csv('raw_data/sepsisexp_timeseries_partition-C.tsv', sep='\t')

df_d = pd.read_csv('raw_data/sepsisexp_timeseries_partition-D.tsv', sep='\t')

print(df_a.head())

      id  sepsis  severity  timestep  respiratory_minute_volume  heart_rate  \
0  12292       0       0.0       0.0                   0.190898    0.424464   
1  12292       0       0.0       0.5                   0.157654    0.667394   
2  12292       0       0.0       1.0                   0.024678    0.618808   
3  12292       0       0.0       1.5                  -0.208030    0.278706   
4  12292       0       0.0       2.0                  -0.108298   -0.352912   

   leukocytes  temperature  partial_co2  respiratory_rate  ...   calcium  \
0    0.301015    -0.168117    -0.275272          1.879692  ...  1.019083   
1    0.301015    -0.168117    -0.275272          1.708485  ...  1.019083   
2    0.301015    -0.732387     1.003408          2.050899  ... -0.868157   
3    0.301015    -0.732387     1.003408          1.366071  ... -0.868157   
4    0.301015    -0.732387     1.094023          1.537278  ... -0.868157   

   potassium  mixed_venous_oxygen_saturation  urine_output  net bala

In [59]:
train_df = pd.concat([df_a, df_b], axis=0).reset_index(drop=True)
val_df = df_c.reset_index(drop=True)
test_df = df_d.reset_index(drop=True)

In [60]:
excluded = ['id', 'sepsis', 'severity', 'timestep']
features = [col for col in train_df.columns if col not in excluded]

In [61]:
def add_targets(df):
    df['target_2h'] = df.groupby('id')['sepsis'].shift(-4)
    df['target_4h'] = df.groupby('id')['sepsis'].shift(-8)
    df['target_6h'] = df.groupby('id')['sepsis'].shift(-12)
    return df

In [62]:
df_train = add_targets(train_df).ffill().fillna(0)
df_val = add_targets(val_df).ffill().fillna(0)
df_test = add_targets(test_df).ffill().fillna(0)

In [63]:
def create_sequences(df, feature_cols, target_col, window_size=12):
    X, y = [], []
    for _, group in df.groupby('id'):
        if len(group) <= window_size:
            continue
        
        data = group[feature_cols].values
        target = group[target_col].values
        
        for i in range(len(group) - window_size):
            X.append(data[i : i + window_size])
            y.append(target[i + window_size])
            
    return np.array(X), np.array(y)

In [64]:
X_train_2h, y_train_2h = create_sequences(df_train, features, 'target_2h', window_size=12)
X_val_2h, y_val_2h = create_sequences(df_val, features, 'target_2h', window_size=12)
X_test_2h, y_test_2h = create_sequences(df_test, features, 'target_2h', window_size=12)

X_train_4h, y_train_4h = create_sequences(df_train, features, 'target_4h', window_size=12)
X_val_4h, y_val_4h = create_sequences(df_val, features, 'target_4h', window_size=12)
X_test_4h, y_test_4h = create_sequences(df_test, features, 'target_4h', window_size=12)

X_train_6h, y_train_6h = create_sequences(df_train, features, 'target_6h', window_size=12)
X_val_6h, y_val_6h = create_sequences(df_val, features, 'target_6h', window_size=12)
X_test_6h, y_test_6h = create_sequences(df_test, features, 'target_6h', window_size=12)

## DENNA MODEL ÄR HEMSK MAX. IDK WHAT TO DO

In [118]:
def train_model(X_train, y_train, X_val, y_val, label_name):
    
    print(f"\n--- Tränar modell för {label_name} ---")
    
    model = Sequential()

    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

    model.add(LSTM(64, return_sequences=False))

    model.add(Dropout(0.4))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    model.compile(optimizer=optimizer, 
        loss='binary_crossentropy', 
        metrics=['accuracy', 'auc'])
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        mode='auto',
        patience=3, 
        restore_best_weights=True)


    model.fit(
        X_train, y_train, 
        validation_data=(X_val, y_val), 
        epochs=25, batch_size=20, 
        callbacks=[early_stop], verbose=1)
    
    return model

In [ ]:
model_2h = train_model(X_train_2h, y_train_2h, X_val_2h, y_val_2h, "2h Advance")

In [107]:
model_4h = train_model(X_train_4h, y_train_4h, X_val_4h, y_val_4h, "4h Advance")


--- Tränar modell för 4h Advance ---
Epoch 1/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.8580 - auc: 0.9514 - loss: 0.5138 - val_accuracy: 0.6988 - val_auc: 0.7209 - val_loss: 1.1686
Epoch 2/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.9504 - auc: 0.9849 - loss: 0.2768 - val_accuracy: 0.6889 - val_auc: 0.7158 - val_loss: 1.2935
Epoch 3/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.9635 - auc: 0.9850 - loss: 0.2642 - val_accuracy: 0.6843 - val_auc: 0.7089 - val_loss: 1.3169
Epoch 4/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.9688 - auc: 0.9863 - loss: 0.2486 - val_accuracy: 0.6915 - val_auc: 0.7157 - val_loss: 1.3604


In [103]:
model_6h = train_model(X_train_6h, y_train_6h, X_val_6h, y_val_6h, "6h Advance")


--- Tränar modell för 6h Advance ---
Epoch 1/25
10041/10041 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - accuracy: 0.7269 - auc: 0.7981 - loss: 0.5319 - val_accuracy: 0.7174 - val_auc: 0.7485 - val_loss: 0.6003
Epoch 2/25
10041/10041 ━━━━━━━━━━━━━━━━━━━━ 28s 3ms/step - accuracy: 0.8077 - auc: 0.8831 - loss: 0.4165 - val_accuracy: 0.7215 - val_auc: 0.7540 - val_loss: 0.6238
Epoch 3/25
10041/10041 ━━━━━━━━━━━━━━━━━━━━ 28s 3ms/step - accuracy: 0.8462 - auc: 0.9193 - loss: 0.3508 - val_accuracy: 0.7141 - val_auc: 0.7502 - val_loss: 0.6771
Epoch 4/25
10041/10041 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - accuracy: 0.8729 - auc: 0.9419 - loss: 0.2991 - val_accuracy: 0.7044 - val_auc: 0.7447 - val_loss: 0.7407


## Task 1: This is the title of task 1

This section should contain the solution of task 1.

It is mandatory to maintain the headings for each task.  
OPTIONALLY, you can use one level down (###) to organize subsessions of the assignments.

Use markdown cells like this one to include:
- Discussion points.
- References to specific sources of code that you might have used to solve the assignment.
- General commentas and explanations about your solution.

In [ ]:
# Always use comments in the code to document specific steps

## Task 2: This is the title of task 2

This section should contain the solution of task 2.

## Results and Discussion

This section should contain:
- Results.
- Summary of best model performance:
    - Name of best model file as saved in /models.
    - Relevant scores such as: accuracy, precision, recall, F1-score, etc.
- Key discussion points.

In [ ]:
# Always use comments in the code to document specific steps